# Tic-Tac-Toe Endgame Classification

958 completed tic-tac-toe boards, each described by the contents of its nine squares (x, o or blank). Predict whether the x player won — a pure logic problem with a known answer, but one that generic classifiers must learn from examples alone.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Tic-Tac-Toe Endgame (id 101)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/101/data.csv')))
print(df.shape)
print('Boards per outcome:')
print(df['class'].value_counts())
df.head(3)


(958, 10)
Boards per outcome:
class
positive    626
negative    332
Name: count, dtype: int64


,top-left-square,top-middle-square,top-right-square,middle-left-square,middle-middle-square,middle-right-square,bottom-left-square,bottom-middle-square,bottom-right-square,class
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive


In [2]:

# All nine features are categorical, so one-hot encode them before splitting.
X = pd.get_dummies(df.drop(columns=['class'])); y = df['class']
feats = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9750
Logistic Regression accuracy: 0.9792
              precision    recall  f1-score   support

    negative       1.00      0.93      0.96        83
    positive       0.96      1.00      0.98       157

    accuracy                           0.97       240
   macro avg       0.98      0.96      0.97       240
weighted avg       0.98      0.97      0.97       240



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Top 20 one-hot board positions')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


middle-middle-square_o    0.084023
middle-middle-square_x    0.071845
top-left-square_o         0.050885
top-left-square_x         0.049067
bottom-left-square_o      0.048776
top-right-square_x        0.048771
bottom-right-square_x     0.048269
bottom-right-square_o     0.048151
top-right-square_o        0.047544
bottom-left-square_x      0.044594
middle-left-square_o      0.040790
bottom-middle-square_x    0.038202
middle-left-square_x      0.037448
middle-right-square_x     0.037222
bottom-middle-square_o    0.036278
top-middle-square_x       0.034929
middle-right-square_o     0.033808
top-middle-square_o       0.032659
top-middle-square_b       0.022012
bottom-middle-square_b    0.021787
dtype: float64


Winner: Logistic Regression
